In [ ]:
!pip install ultralytics

In [ ]:
import kagglehub
from pathlib import Path
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
import matplotlib.patches as patches
from glob import glob
import random
import os
import shutil
from ultralytics import YOLO
import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np

# > One-stage detection. YOLO

## Описание задачи  
В этом задании мы продолжим практиковаться в детекции и сегментации людей на примере того же датасета, что мы использовали в прошлый раз, но в этот раз мы будем использовать модели YOLO вместо Mask-RCNN.

## План
- Подготовка датасета для задачи детекции
- Обучение модели YOLO для задачи детекции
- Подготовка датасета для задачи сегментации
- Обучение модели YOLO для задачи сегментации



# > Подготовка датасета для задачи детекции

### Подготовка bbox-ов
Можем воспользоваться ранее скачанным датасетом или скачать его заново.



In [ ]:
dataset_path = kagglehub.dataset_download("tapakah68/segmentation-full-body-mads-dataset")

Мы уже работали с этим датасетом, но давайте ещё раз посмотрим, как он выглядит.

In [ ]:
image_paths = Path(dataset_path).rglob(f'collages/*.jpg')
n = 5
fig, axes = plt.subplots(n, 1, figsize=(8, 2 * n))

for i, path in enumerate(image_paths):
  if i >= n:
    break
  image = read_image(path)
  axes[i].imshow(image.permute(1, 2, 0))
  axes[i].axis('off')
plt.tight_layout()
plt.show()

Как вы могли заметить, координат bbox-ов у нас нет, как и в прошлый раз, мы получим их из масок, но теперь нам нужно будет преобразовать их в YOLO-формат.


**Задание 1**. реализуйте функцию `get_boxes` для получения координат bbox-ов из имеющихся масок, используя `torchvision.ops.boxes.masks_to_boxes`.

In [ ]:
def get_boxes(path_to_masks: list) -> dict:
  """
  Извлекает bbox-ы из масок.

  :param path_to_masks: список путей с изображениями масок
  :return image_name_to_boxes:  словарь, сопоставляющий имя файла и соответствующий ему bbox
  """
  image_name_to_boxes = {}
  for mask_path in path_to_masks:
    image_name = Path(mask_path).stem
    mask = read_image(mask_path)
    bbox = ...
    image_name_to_boxes[image_name] = bbox[0]
  return image_name_to_boxes


Мы получили координаты bbox-ов в формате **Pascal VOC**:
```
[xmin, ymin, xmax, ymax]
```
Давайте для начала посмотрим, как выглядят координаты, которые мы получили.

In [ ]:
def visualize_bounding_box(image_tensor, bbox):
    image_np = image_tensor.permute(1, 2, 0).numpy()  # (H, W, C)
    fig, ax = plt.subplots(1)
    ax.imshow(image_np)

    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    # рисуем bbox
    rect = patches.Rectangle((xmin, ymin), width, height, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)


    # Добавляем точки на изображение
    ax.scatter(xmin, ymin, color='white', s=10, marker='o')
    ax.scatter(xmax, ymax, color='white', s=10, marker='o')
    ax.text(xmin, ymin, f'[{xmin}, {ymin}]', fontsize=10, color='white', ha='right', va='bottom')
    ax.text(xmax, ymax, f'[{xmax}, {ymax}]', fontsize=10, color='white', ha='left', va='top')

    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')

    plt.show();

path_to_images = list(Path(dataset_path).rglob(f'images/*.png'))
path_to_masks = list(Path(dataset_path).rglob(f'masks/*.png'))

image_name_to_boxes = get_boxes(path_to_masks )
image_path = path_to_images[0]
image = read_image(image_path)
image_name = Path(image_path).stem
bbox = image_name_to_boxes[image_name]
print('Pascal VOC формат:\n[xmin, ymin, xmax, ymax]\n')
print('bbox:\n', bbox.tolist())
visualize_bounding_box(image , bbox)

Для обучения модели нам понадобится перевести их в **YOLO**-формат:
```
[x_center/image_width, y_center/image_height, bbox_width/image_width, bbox_height/image_height]
```

**Задание 2.** Реализуйте функцию `pascal_voc_to_yolo_boxes`, которая на вход получает изображение и координаты одного bbox-a, а на выход выдаёт координаты в формате YOLO `(x_center/w, y_center/h, width/w, height/h)`.

In [ ]:
def pascal_voc_to_yolo_boxes(image_path, bbox):
  """
  Конвертирует bbox-ы из формата Pascal VOC в формат YOLO.

  :param image_path: путь к изображению, для которого предназначен bbox
  :param bbox: bbox в формате Pascal VOC, содержащий координаты (xmin, ymin, xmax, ymax)
  :return yolo_bbox: bbox в формате YOLO, содержащий нормализованные координаты центра рамки и её размеры: [x_center_norm, y_center_norm, width_norm, height_norm]
  """
  image = read_image(image_path)
  ..., ..., ... = image.shape
  ..., ..., ..., ... = bbox
  h = ...
  w = ...
  x_center = ...
  y_center = ...

  x_center_norm = ...
  y_center_norm =  ...
  width_norm = ...
  height_norm = ...
  yolo_bbox = [x_center_norm.item(), y_center_norm.item(), width_norm.item(), height_norm.item()]

  return yolo_bbox


image_name = Path(image_path).stem
bbox = image_name_to_boxes[image_name]
yolo_box = pascal_voc_to_yolo_boxes(image_path, bbox)


На примере предыдущего изображения координаты bbox-а  в формате YOLO будут иметь такой вид:

In [ ]:
def visualize_yolo_bbox(image_tensor, bbox):
    print('YOLO формат\n(x_center_normalize, y_center_normalize, width_normalize, height_normalize)')
    print('bbox:\n', bbox)
    # Переводим изображение из тензора в numpy-массив
    image_np = image_tensor.permute(1, 2, 0).numpy()  # (H, W, C)
    fig, ax = plt.subplots(1)
    ax.imshow(image_np)

    # Получаем размеры изображения
    _, h, w = image.shape
    # Декодируем координаты YOLO
    x_center, y_center, width, height = bbox

    # Преобразуем нормализованные координаты в пиксели
    x_center_pix = x_center * w
    y_center_pix = y_center * h
    width_pix = width * w
    height_pix = height * h

    # Вычисляем верхний левый угол и нижний правый угол
    xmin = int(x_center_pix - width_pix / 2)
    ymin = int(y_center_pix - height_pix / 2)
    xmax = int(x_center_pix + width_pix / 2)
    ymax = int(y_center_pix + height_pix / 2)

    # # Рисуем прямоугольник на изображении
    # plt.imshow(image)
    # plt.axis('off')  # Отключаем оси
      # Рисуем bbox
    rect = patches.Rectangle((xmin, ymin), width_pix, height_pix, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)


    # Визуализируем центр
    ax.scatter(x_center_pix, y_center_pix, color='white', s=10, marker='o')
    plt.text(x_center_pix + 5, y_center_pix, f'({round(x_center,2)}, {round(y_center,2)})', fontsize=10, color='white', ha='right', va='bottom')

    # # Добавляем текст для высоты и ширины
    plt.text(x_center_pix - 30, ymax + 15, f"Width: {width:.2f}", color='white', fontsize=10)
    plt.text(xmax + 15, y_center_pix, f"Height: {height:.2f}", color='white', fontsize=10)

    # Устанавливаем относительные оси от 0 до 1
    ax.set_xticks(np.linspace(0, w, num=9))  # 6 меток по оси X
    ax.set_yticks(np.linspace(0, h, num=10))  # 6 меток по оси Y

    # Настраиваем метки для относительных значений
    ax.set_xticklabels(np.linspace(0, 1, num=9), rotation=0)
    ax.set_yticklabels(np.round(np.linspace(0, 1, num=10), 2))  # Переворот для правильного отображения Y
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    # Показываем изображение
    plt.show()

# Пример использования

visualize_yolo_bbox(image, yolo_box)


### Разделение данных на train и val

Теперь нам нужно перевести данные в формат, необходимый для обучения модели.
Для этого нужно подготовить изображения и разметку bbox-ов в определённом формате и указать пути к ним в YAML-конфиге.

```
dataset_path
├── images
│   ├── train
│   │   ├── image_1.jpg
│   │   ├── image_2.jpg
│   └── val
│       ├── image_3.jpg
│       └── image_4.jpg
└── labels
    ├── train
    │   ├── image_1.txt
    │   └── image_2.txt
    └── val
        ├── image_3.txt
        └── image_4.txt
```
Файлы с разметкой имеют следующий формат:
```
0 0.568359375 0.578125 0.33984375 0.64583331
```
Вначале идёт индекс класса, в нашей задаче только один класс 'person' с индексом 0. Затем идут координаты bbox-ов в YOLO-формате.

Затем нужно в YAML-конфиге указать путь к данным. Конфиг выглядит следующим образом:
```
path: PATH/TO/DATASET # dataset root dir
train: images/train # train images (relative to 'path') 4 images
val: images/val # val images (relative to 'path') 4 images
test: # test images (optional)

# Classes
names:
  0: person
```


Для начала разобьём данные на train и val.

**Задание 3.** Реализуйте функцию train_val_split. На вход она принимает путь к папке с изображениями и выдаёт два списка, пути к train- и val-файлам.


In [ ]:
def train_val_split(image_paths , validation_percentage):
    """
    Разделяет список путей к изображениям на обучающую и валидационную выборки.

    :param image_paths: список путей к изображениям
    :param validation_percentage: процент данных, выделяемый для валидационной выборки (значение от 0 до 1)
    :return: кортеж, содержащий:
            - training_paths: список путей к изображениям, используемым для обучения;
            - validation_paths: список путей к изображениям, используемым для валидации
    """
    ...
    return training_paths, validation_paths

image_paths = list(Path(dataset_path).rglob('*/images/*.png'))
training_paths, validation_paths = train_val_split(image_paths, validation_percentage=0.1)

### Подготовка структуры папок для хранения датасета

Мы разбили данные на train и val, теперь разложим всё по папкам и создадим YAML-конфиг.

**Задание 4.** Реализуйте функцию create_dataset_template. На вход она принимает путь к папке, где будет создан шаблон для датасета. Функция должна создать следующую структуру папок:

In [ ]:
def create_dataset_template(path_to_create_dataset_template):
  """
  Создаёт шаблон структуры для хранения датасета с папками для тренировочных и валидационных изображений и меток.

  :param path_to_create_dataset_template: путь, по которому будет создана структура папок для датасета
  :return: кортеж с абсолютными путями к созданным папкам:
           - absolute_path_to_dataset: путь к основной папке датасета;
           - absolute_path_to_train_images: путь к папке с тренировочными изображениями;
           - absolute_path_to_train_labels: путь к папке с тренировочными метками;
           - absolute_path_to_val_images: путь к папке с валидационными изображениями;
           - absolute_path_to_val_labels: путь к папке с валидационными метками
  """
  ...
  return absolute_path_to_dataset, absolute_path_to_train_images, absolute_path_to_train_labels, absolute_path_to_val_images, absolute_path_to_val_labels


path_to_create_dataset_template = '/opt/dataset' # создадим датасет в текущей папке, но вы здесь можете указать любую
path_to_dataset, path_to_train_images, path_to_train_labels, path_to_val_images, path_to_val_labels = create_dataset_template(path_to_create_dataset_template)

### Создание датасета для задачи детекции

Ранее мы подготовили словарь image_name_to_boxes, сопоставляющий имя файла с координатами bbox-а. Сейчас он нам пригодится. Нам нужно разложить train- и val-файлы по соответствующим папкам и сформировать там же файлы с разметкой.

**Задание 5.** Реализуйте функцию prepare_dataset, на вход она получает список файлов, путь, куда будет сохранён датасет, и словарь с координатами bbox-ов. Функция должна скопировать файлы из input_paths в path_to_save_images и создать в папке path_to_save_labels txt-файлы с разметкой в формате:

class_id x_center_normalize, y_center_normalize, width_normalize, height_normalize



In [ ]:
def prepare_dataset(input_paths, path_to_save_images, path_to_save_labels, image_name_to_boxes):
  """
  Подготавливает датасет, копируя изображения и создавая YOLO-аннотации из bbox-ов в формате Pascal VOC.

  :param input_paths: список путей к изображениям, которые нужно обработать
  :param path_to_save_images: путь, куда будут сохранены копии изображений
  :param path_to_save_labels: путь, куда будут сохранены аннотации в формате YOLO
  :param image_name_to_boxes: словарь, сопоставляющий имя изображения и соответствующие рамки в формате Pascal VOC
  """
  for image_path in input_paths:
    # copy image_path to path_to_save_images
    image_name = image_path.stem
    pascal_voc_boxes = image_name_to_boxes[image_name]
    yolo_boxes = pascal_voc_to_yolo_boxes(image_path, pascal_voc_boxes)
    ...
    yolo_boxes = ...
    path_to_annotation = Path(path_to_save_labels, f'{image_name}.txt')
    with open(path_to_annotation, 'w') as f_out:
      f_out.write(yolo_boxes)


prepare_dataset(training_paths, path_to_train_images, path_to_train_labels, image_name_to_boxes)
prepare_dataset(validation_paths, path_to_val_images, path_to_val_labels, image_name_to_boxes)

### Создание YAML-конфига для описания датасета

Данные мы подготовили, теперь нужно подготовить YAML-конфиг.

**Задание 6**. Реализуйте функцию для подготовки YAML-конфига. На вход функция принимает абсолютные пути до датасета, и к папкам с train- и val-изображениями.



In [ ]:
import yaml

def generate_yaml_config(dataset_path, path_to_train, path_to_val):
    """
    Генерирует YAML-конфиг для датасета, содержащий относительные пути к тренировочной и валидационной выборкам, а также карту имён классов.

    :param dataset_path: путь к основной папке датасета
    :param path_to_train: абсолютный путь к папке с тренировочными данными
    :param path_to_val: абсолютный путь к папке с валидационными данными
    :return: абсолютный путь к созданному YAML конфигурационному файлу
    """
    config = {
        'path': str(...), # абсолютный путь к датасету
        'train': str(...), # путь к train-изображениям относительно dataset_path
        'val': str(...), # # путь к val-изображениям относительно dataset_path
        'names': {
            '0': 'person'
            },

    }
    path_to_config = Path('config.yaml')
    with open('config.yaml', 'w') as file:
        yaml.dump(config, file)
    absolute_path_to_config = path_to_config.resolve()
    return

generate_yaml_config(path_to_dataset, path_to_train_images, path_to_val_images)

# > Обучение модели YOLO для задачи детекции
Cамый сложный этап мы прошли. Когда данные готовы, нам остаётся только запустить обучение. Для этого нам нужно скачать конфиг модели.

In [ ]:
!wget https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v8/yolov8.yaml


Давайте создадим модель из конфига и поставим модель обучаться.

In [ ]:
model = YOLO('yolov8.yaml')


In [ ]:
train_results = model.train(data='config.yaml', epochs=10)

In [ ]:
val_results = model.val()

In [ ]:
path_to_or_curve = Path(train_results.save_dir, 'results.png')
pr_curve= cv2.imread(path_to_or_curve)
plt.figure(figsize=(15, 8))
plt.imshow(pr_curve[:, :, ::-1])
plt.show()

In [ ]:
val_images_paths = list(path_to_val_images.rglob('*.png'))
i = random.randint(0,len(val_images_paths))
image_path = val_images_paths[i]
image = cv2.imread(image_path)
results = model.predict(source=image, save=True)
res = cv2.imread(Path(results[0].save_dir, results[0].path))
plt.imshow(res[:, :, ::-1])
plt.show()


**Задание 7**. Конвертируйте обученную модель в ONNX и загрузите файл с моделью.

In [ ]:
onnx_path = ...

# > Подготовка датасета для задачи сегментации
### Подготовка масок

Давайте теперь попробуем решить задачу сегментации.
Для этого нам нужно будет преобразовать имеющиеся у нас маски в формат, похожий на тот, что мы использовали при обучении детекции.


> cv2.findContours может найти больше одного контура на изображение с маской, возьмите контур с максимальной площадью.


In [ ]:
def extract_polygon_points(mask_path):
    """
    Извлекает координаты многоугольника, описывающего маску, нормализованные по ширине и высоте изображения.

    :param mask_path: путь к изображению маски
    :return: список нормализованных координат  многоугольника, описывающего маску
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    height, width = mask.shape

    # Извлечение контуров
    contours, _ = ...
    max_contour = ...
    polygon_points = []
    for point in contour:
        x_norm = ...
        y_norm = ...
        polygon_points.extend([x_norm, y_norm])
    return



### Создание датасета для задачи сегментации

Так же, как мы делали для детекции, cоздадим словарь, сопоставляющий имя файла с координатами маски.



In [ ]:
path_to_masks = list(Path(dataset_path).rglob(f'masks/*.png'))

image_name_to_mask = {}
for mask_path in path_to_masks[:]:
  name = mask_path.stem
  mask_points = extract_polygon_points(mask_path)
  image_name_to_mask[name] = mask_points


Осталось создать датасет, и можно будет начать обучение.

**Задание 9**. Реализуйте функцию `prepare_dataset` для создания датасета, как и в случае детекции, результатом функции будет созданный датасет. Для простоты мы просто заменили разметку для детекции на разметку сегментации.

In [ ]:
def prepare_dataset(input_paths, path_to_save_images, path_to_save_labels, image_name_to_mask ):
  """
  Подготавливает датасет, копируя изображения и создавая файлы разметки с координатами, описывающими маски.

  :param input_paths: список путей к изображениям, которые нужно обработать
  :param path_to_save_images: путь, куда будут сохранены копии изображений
  :param path_to_save_labels: путь, куда будут сохранены аннотации с точками маски
  :param image_name_to_mask: словарь, сопоставляющий имя изображения и соответствующие точки маски
  """
  for image_path in input_paths:
    shutil.copy(image_path, path_to_save_images)
    image_name = image_path.stem
    mask_points = image_name_to_mask[image_name]
    mask_str = ...
    path_to_annotation = Path(path_to_save_labels, f'{image_name}.txt')
    with open(path_to_annotation, 'w') as f_out:
      f_out.write(mask_str)

prepare_dataset(training_paths, path_to_train_images, path_to_train_labels, image_name_to_mask )
prepare_dataset(validation_paths, path_to_val_images, path_to_val_labels, image_name_to_mask )

### Выбор размера модели
Теперь всё готово для обучения модели сегментации. Скачаем конфигурацию модели и инициализируем модель.

In [ ]:
!wget https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v8/yolov8-seg.yaml




При обучении детектора мы никак не изменяли исходный конфиг модели. Вы, скорее всего, могли заметить предупреждение:
> WARNING ⚠️ no model scale passed. Assuming scale='n'.

 В конфиге модели есть 5 вариантов её размера, по умолчанию используется самая маленькая — n.

 ```
 scales: # model compound scaling constants, i.e. 'model=yolov8n-seg.yaml' will call yolov8-seg.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

```

**Задание 10**. Реализуйте функцию `set_scale`. На вход функция принимает YAML-конфиг и требуемый размер модели (n/s/m/l/x). В результате функция добавляет в конфиг поле `'scale'` с заданным значением. Итоговая конфигурация будет иметь такой вид:
```
...
scale: n
scales: # model compound scaling constants, i.e. 'model=yolov8n-seg.yaml' will call yolov8-seg.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

```

In [ ]:
def set_sacle(config_path, scale):
  """
  Обновляет конфигурационный файл YAML, устанавливая новое значение 'scale'.

  :param config_path: путь к конфигурационному файлу YAML
  :param scale: новое значение размера модели, которое нужно установить
  """
  ...



# > Обучение модели YOLO для задачи сегментации

Для решения нашей задачи нам достаточно самой маленькой модели, но в этот раз давайте явно это зададим.



In [ ]:
config_path = 'yolov8-seg.yaml'
set_scale(config_path, scale='n')

Запустим обучение модели. 10 эпох должно хватить, чтобы обучить неплохую модель.

In [ ]:
model = YOLO('yolov8n-seg.yaml')

In [ ]:
train_results = model.train(data='config.yaml', epochs=10)

**Задание 11**. Конвертируйте обученную модель в ONNX и загрузите файл с моделью.

In [ ]:
onnx_path = ...